# 02 Baselines and models (validation only)

Do not open any test result in this notebook. The test set stays untouched until Week 3.

In [ ]:
import json, os, sys
from pathlib import Path

# Work from the repository root whether the notebook is opened from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cfg = json.load(open("configs/dataset.json"))
OUT = Path("results/data_quality"); OUT.mkdir(parents=True, exist_ok=True)
cfg

## Split overview

In [ ]:
split = json.load(open("configs/split.json"))
pd.DataFrame(split["splits"]).T

## Baselines on validation

In [ ]:
from baselines import hour_of_week_predict, persistence_predict
train = pd.read_parquet("data/processed/train.parquet")
val = pd.read_parquet("data/processed/val.parquet")
mae = lambda y, p: float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))
print("persistence  val MAE:", round(mae(val["target_2h"], persistence_predict(val)), 3))
print("hour-of-week val MAE:", round(mae(val["target_2h"], hour_of_week_predict(train, val)), 3))

## Candidate models on validation (after running `python src/03_train_select.py`)

In [ ]:
vm = pd.read_csv("results/validation_metrics.csv").sort_values("val_mae")
ax = vm.plot.barh(x="model", y="val_mae", legend=False, figsize=(7, 4)); ax.invert_yaxis()
ax.set_xlabel("validation MAE (km/h)"); ax.set_title("Validation MAE by model")
plt.tight_layout(); plt.savefig("results/validation_mae.png", dpi=150); plt.show()
vm.round(3)

## Tuned settings

In [ ]:
final = json.load(open("configs/final_model.json"))
print("Selected on validation:", final["selected"], "| ensemble members:", final["ensemble_members"])
pd.Series({k: str(v) for k, v in final["params"].items()}, name="best params")

## Notes
Write here, in your own words: why the winning model won on validation, and which models did not beat the hour-of-week baseline.